# Notebook 6: 2D Grids and Matrix Operations

Many real-world problems are naturally 2D (images, matrices). In this notebook, you will learn:

- Using `dim3` for 2D grid and block dimensions
- 2D thread indexing for matrices
- Row-major memory layout
- Matrix addition, scaling, and transpose
- Correct bounds checking in 2D

In [ ]:
%load_ext nvcc4jupyter

## 6.1 Row-Major Layout

C/C++ stores matrices in **row-major** order: rows are contiguous in memory.

```
Matrix (3x4):          Memory layout:
┌─────────────┐        [a00 a01 a02 a03 | a10 a11 a12 a13 | a20 a21 a22 a23]
│ 0  1  2  3  │         row 0            row 1              row 2
│ 4  5  6  7  │
│ 8  9  10 11 │        Index formula: element[row][col] = data[row * width + col]
└─────────────┘
```

This formula is fundamental. You'll use it in every 2D kernel.

## 6.2 2D Thread Indexing with `dim3`

`dim3` specifies grid and block dimensions in up to 3 dimensions:

```cpp
dim3 block(16, 16);        // 16x16 = 256 threads per block
dim3 grid(ceil(W/16), ceil(H/16));  // Enough blocks to cover the matrix
kernel<<<grid, block>>>(...);
```

Inside the kernel:
```cpp
int col = blockIdx.x * blockDim.x + threadIdx.x;
int row = blockIdx.y * blockDim.y + threadIdx.y;
```

In [ ]:
%%cuda
#include <cstdio>

__global__ void show_2d_indices() {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // Only print a small region to keep output readable
    if (row < 4 && col < 4) {
        printf("Thread (row=%d, col=%d) | block=(%d,%d) thread=(%d,%d)\n",
               row, col, blockIdx.x, blockIdx.y, threadIdx.x, threadIdx.y);
    }
}

int main() {
    // 2x2 grid of 2x2 blocks
    dim3 block(2, 2);
    dim3 grid(2, 2);

    printf("Grid: (%d, %d) blocks  Block: (%d, %d) threads\n",
           grid.x, grid.y, block.x, block.y);
    printf("Total threads: %d x %d = %d\n\n",
           grid.x * block.x, grid.y * block.y,
           grid.x * block.x * grid.y * block.y);

    show_2d_indices<<<grid, block>>>();
    cudaDeviceSynchronize();
    return 0;
}

### Convention: x = column, y = row

CUDA's convention: `threadIdx.x` varies fastest (like columns), `threadIdx.y` is the row. This matches how threads are physically arranged in memory for coalesced access.

```
dim3 block(TILE_W, TILE_H);
      ↓ col   ↓ row
```

## 6.3 Matrix Addition

The simplest 2D operation: C[i][j] = A[i][j] + B[i][j].

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

__global__ void matrix_add(const float* A, const float* B, float* C,
                           int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < rows && col < cols) {
        int idx = row * cols + col;  // Row-major index
        C[idx] = A[idx] + B[idx];
    }
}

void print_matrix(const char* name, const float* M, int rows, int cols) {
    printf("%s:\n", name);
    for (int r = 0; r < rows; r++) {
        printf("  [");
        for (int c = 0; c < cols; c++) {
            printf("%5.1f", M[r * cols + c]);
            if (c < cols - 1) printf(", ");
        }
        printf("]\n");
    }
}

int main() {
    const int ROWS = 4, COLS = 5;
    const size_t bytes = ROWS * COLS * sizeof(float);

    // Host
    float* h_A = (float*)malloc(bytes);
    float* h_B = (float*)malloc(bytes);
    float* h_C = (float*)malloc(bytes);

    for (int i = 0; i < ROWS * COLS; i++) {
        h_A[i] = (float)i;
        h_B[i] = (float)(i * 10);
    }

    // Device
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));

    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));

    // Launch with 2D grid
    dim3 block(16, 16);  // 256 threads per block
    dim3 grid((COLS + block.x - 1) / block.x,
              (ROWS + block.y - 1) / block.y);

    printf("Grid: (%d, %d)  Block: (%d, %d)\n\n", grid.x, grid.y, block.x, block.y);

    matrix_add<<<grid, block>>>(d_A, d_B, d_C, ROWS, COLS);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost));

    print_matrix("A", h_A, ROWS, COLS);
    print_matrix("B", h_B, ROWS, COLS);
    print_matrix("C = A + B", h_C, ROWS, COLS);

    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaFree(d_C));
    free(h_A); free(h_B); free(h_C);
    return 0;
}

## 6.4 Matrix Transpose

Transpose: `B[j][i] = A[i][j]`. The output has swapped dimensions.

```
A (3x4):           B = A^T (4x3):
┌─────────────┐    ┌──────────┐
│ 0  1  2  3  │    │ 0  4  8  │
│ 4  5  6  7  │ →  │ 1  5  9  │
│ 8  9  10 11 │    │ 2  6  10 │
└─────────────┘    │ 3  7  11 │
                   └──────────┘
```

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

// Naive transpose: reads are coalesced, writes are strided
__global__ void transpose_naive(const float* A, float* B,
                                int A_rows, int A_cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;  // A's column
    int row = blockIdx.y * blockDim.y + threadIdx.y;  // A's row

    if (row < A_rows && col < A_cols) {
        // Read from A[row][col], write to B[col][row]
        B[col * A_rows + row] = A[row * A_cols + col];
    }
}

void print_matrix(const char* name, const float* M, int rows, int cols) {
    printf("%s (%dx%d):\n", name, rows, cols);
    for (int r = 0; r < rows; r++) {
        printf("  [");
        for (int c = 0; c < cols; c++) {
            printf("%4.0f", M[r * cols + c]);
            if (c < cols - 1) printf(", ");
        }
        printf("]\n");
    }
}

int main() {
    const int ROWS = 3, COLS = 5;
    const size_t bytes = ROWS * COLS * sizeof(float);

    float* h_A = (float*)malloc(bytes);
    float* h_B = (float*)malloc(bytes);  // B is COLS x ROWS

    for (int i = 0; i < ROWS * COLS; i++) h_A[i] = (float)i;

    float *d_A, *d_B;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));

    dim3 block(16, 16);
    dim3 grid((COLS + block.x - 1) / block.x,
              (ROWS + block.y - 1) / block.y);

    transpose_naive<<<grid, block>>>(d_A, d_B, ROWS, COLS);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(h_B, d_B, bytes, cudaMemcpyDeviceToHost));

    print_matrix("A", h_A, ROWS, COLS);
    print_matrix("A^T", h_B, COLS, ROWS);

    // Verify
    int errors = 0;
    for (int r = 0; r < ROWS; r++)
        for (int c = 0; c < COLS; c++)
            if (h_B[c * ROWS + r] != h_A[r * COLS + c]) errors++;

    printf("\nVerification: %s\n", errors == 0 ? "PASSED" : "FAILED");

    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    free(h_A); free(h_B);
    return 0;
}

### Why is this "naive"?

The naive transpose has **uncoalesced writes**: adjacent threads write to non-adjacent memory locations (stride = A_rows). We'll fix this with shared memory in intermediate-level notebooks.

## 6.5 Matrix Scaling with Performance Measurement

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

__global__ void matrix_scale(float* M, float scalar, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < rows && col < cols) {
        int idx = row * cols + col;
        M[idx] *= scalar;
    }
}

int main() {
    const int ROWS = 4096, COLS = 4096;
    const size_t N = (size_t)ROWS * COLS;
    const size_t bytes = N * sizeof(float);
    const float scalar = 2.5f;

    printf("Matrix: %d x %d = %.1f M elements (%.1f MB)\n",
           ROWS, COLS, N / 1e6, bytes / 1e6);

    float* h_M = (float*)malloc(bytes);
    for (size_t i = 0; i < N; i++) h_M[i] = 1.0f;

    float* d_M;
    CUDA_CHECK(cudaMalloc(&d_M, bytes));
    CUDA_CHECK(cudaMemcpy(d_M, h_M, bytes, cudaMemcpyHostToDevice));

    dim3 block(32, 8);  // 256 threads, wide in x for coalescing
    dim3 grid((COLS + block.x - 1) / block.x,
              (ROWS + block.y - 1) / block.y);

    printf("Block: (%d, %d)  Grid: (%d, %d)\n", block.x, block.y, grid.x, grid.y);

    // Warmup
    matrix_scale<<<grid, block>>>(d_M, 1.0f, ROWS, COLS);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Timed run
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matrix_scale<<<grid, block>>>(d_M, scalar, ROWS, COLS);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);

    // Bandwidth: read + write each element
    float gb = 2.0f * bytes / 1e9;
    printf("\nKernel time: %.3f ms\n", ms);
    printf("Effective bandwidth: %.2f GB/s\n", gb / (ms / 1000.0f));

    // Verify
    CUDA_CHECK(cudaMemcpy(h_M, d_M, bytes, cudaMemcpyDeviceToHost));
    float expected = scalar;  // 1.0 * scalar
    int errors = 0;
    for (size_t i = 0; i < N; i++)
        if (fabsf(h_M[i] - expected) > 1e-5) errors++;
    printf("Verification: %s\n", errors == 0 ? "PASSED" : "FAILED");

    cudaEventDestroy(start); cudaEventDestroy(stop);
    CUDA_CHECK(cudaFree(d_M));
    free(h_M);
    return 0;
}

### Block shape for 2D kernels

For 2D kernels accessing row-major matrices, prefer **wider blocks in x** (the column dimension):

```cpp
dim3 block(32, 8);   // Good: x threads are contiguous in memory
dim3 block(8, 32);   // Worse: fewer threads per memory row
dim3 block(16, 16);  // OK: balanced, common choice
```

This ensures **memory coalescing**: adjacent threads access adjacent memory locations.

## 6.6 Flattened 1D Kernel vs 2D Kernel

For element-wise operations, you can use either 1D or 2D indexing. They produce the same result.

In [ ]:
%%cuda
#include <cstdio>

// Approach 1: 2D indexing
__global__ void add_2d(const float* A, const float* B, float* C,
                       int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < rows && col < cols) {
        int idx = row * cols + col;
        C[idx] = A[idx] + B[idx];
    }
}

// Approach 2: Treat matrix as a flat array
__global__ void add_1d(const float* A, const float* B, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        C[idx] = A[idx] + B[idx];
    }
}

int main() {
    // For element-wise ops, both approaches work identically.
    // Use 2D when the kernel needs to know row/col (transpose, convolution).
    // Use 1D for pure element-wise ops (simpler).

    int rows = 1024, cols = 1024;
    int N = rows * cols;

    printf("2D kernel: dim3 block(16,16), dim3 grid(%d,%d)\n",
           (cols + 15) / 16, (rows + 15) / 16);
    printf("1D kernel: block=256, grid=%d\n", (N + 255) / 256);
    printf("Both process %d elements — same result!\n", N);

    return 0;
}

## 6.7 Key Takeaways

1. **`dim3`** lets you specify 2D (or 3D) grid and block dimensions
2. **Row-major indexing:** `idx = row * cols + col`
3. **2D bounds check:** `if (row < rows && col < cols)` — check BOTH dimensions
4. **x = column, y = row** by CUDA convention
5. **Prefer wider blocks in x** (`dim3(32, 8)`) for row-major data coalescing
6. Use 2D indexing when the kernel needs row/col info; use 1D for element-wise ops
7. **Naive transpose** has uncoalesced writes — shared memory fixes this (intermediate topic)

## Exercises

**Exercise 6.1:** Write a kernel that initializes a matrix with `M[r][c] = r * cols + c` (filling it with linear indices). Verify by printing.

**Exercise 6.2:** Implement element-wise matrix multiplication: `C[i][j] = A[i][j] * B[i][j]` (Hadamard product). Test with 1024x1024 matrices.

**Exercise 6.3:** Write a kernel that computes row sums: for each row `r`, `sums[r] = sum(M[r][0..cols-1])`. Hint: use 1D indexing where each thread handles one row and loops over columns.

In [ ]:
%%cuda
// Exercise 6.1 — Fill matrix with linear indices
#include <cstdio>
#include <cstdlib>

__global__ void fill_indices(float* M, int rows, int cols) {
    // TODO: M[r][c] = r * cols + c
}

int main() {
    const int ROWS = 4, COLS = 6;
    // TODO: Allocate, launch, copy back, print
    return 0;
}

In [ ]:
%%cuda
// Exercise 6.3 — Row sums
#include <cstdio>
#include <cstdlib>

// Each thread computes one row's sum
__global__ void row_sums(const float* M, float* sums, int rows, int cols) {
    // TODO: thread idx = row, loop over columns
}

int main() {
    const int ROWS = 4, COLS = 5;
    // TODO: Fill M with 1.0, compute row sums, verify each = COLS
    return 0;
}

---
**Next:** [Notebook 7 — Performance Measurement](07_Performance_Measurement.ipynb) — Timing, bandwidth, and profiling basics.